# Online Course Recommendation — Group 16

**Duy Tan University, Da Nang — DS 423**  
**Trần Lãnh:** 28211151726 · **Trịnh Minh Son:** 28211144373

This notebook contains the complete verified content-based recommender and its evaluation workflow.

## 1. Method overview

1. Combine course title, skills, provider, difficulty, and description.
2. Fit TF-IDF vocabulary and IDF weights.
3. Compute cosine similarity between a query and the catalog.
4. Rank with 95% content similarity and a 5% rating tie-breaker.
5. Return the Top-N courses.

> This is not supervised training: there is no target label, loss function, epoch, or gradient descent. `fit_transform()` learns the TF-IDF vocabulary and IDF weights.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

DATA_PATH = Path('Coursera_cleaned_for_recommendation.csv')
RANDOM_STATE = 42

## Core recommender functions

These functions make the notebook fully standalone; no external source file is required.

In [ ]:
def load_courses(path):
    df = pd.read_csv(path)
    required = {
        'course_id', 'course_title', 'university', 'difficulty_level',
        'course_rating_filled', 'course_url', 'course_description', 'skills'
    }
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    text_cols = [
        'course_title', 'university', 'difficulty_level',
        'course_description', 'skills'
    ]
    df[text_cols] = df[text_cols].fillna('')
    return df.drop_duplicates('course_id').reset_index(drop=True)


def build_text(df):
    return (
        (df['course_title'] + ' ') * 3
        + (df['skills'] + ' ') * 2
        + df['university'] + ' '
        + df['difficulty_level'] + ' '
        + df['course_description']
    ).str.lower()


def fit_model(df, max_features=30_000):
    vectorizer = TfidfVectorizer(
        stop_words='english', ngram_range=(1, 2), min_df=2,
        max_df=0.95, max_features=max_features, sublinear_tf=True
    )
    matrix = vectorizer.fit_transform(build_text(df))
    return vectorizer, matrix

In [ ]:
def recommend_by_index(df, matrix, index, n=10, rating_weight=0.05):
    similarity = linear_kernel(matrix[index], matrix).ravel()
    rating = df['course_rating_filled'].astype(float).to_numpy()
    rating_scaled = np.clip((rating - 1.0) / 4.0, 0, 1)
    score = (1 - rating_weight) * similarity + rating_weight * rating_scaled
    score[index] = -np.inf
    top = np.argpartition(score, -n)[-n:]
    top = top[np.argsort(score[top])[::-1]]
    columns = [
        'course_id', 'course_title', 'university', 'difficulty_level',
        'course_rating_filled', 'course_url'
    ]
    result = df.loc[top, columns].copy()
    result.insert(0, 'rank', range(1, len(result) + 1))
    result['score'] = score[top].round(4)
    return result.reset_index(drop=True)


def recommend_by_title(df, matrix, title, n=10):
    exact = df.index[df['course_title'].str.casefold() == title.casefold()].tolist()
    if exact:
        index = exact[0]
    else:
        contains = df.index[df['course_title'].str.contains(
            re.escape(title), case=False, regex=True
        )].tolist()
        if not contains:
            raise KeyError(f'Course title not found: {title}')
        index = contains[0]
    return recommend_by_index(df, matrix, index, n=n)

In [ ]:
def skill_sets(df):
    return [
        {s.strip().lower() for s in str(value).split(',') if s.strip()}
        for value in df['skills']
    ]


def evaluate(df, matrix, ks=(5, 10, 20), sample_size=250,
             relevance_threshold=0.20):
    rng = np.random.default_rng(RANDOM_STATE)
    skills = skill_sets(df)
    candidates = [i for i, values in enumerate(skills) if len(values) >= 2]
    query_ids = rng.choice(
        candidates, size=min(sample_size, len(candidates)), replace=False
    )
    rows, example = [], {}
    max_k = max(ks)
    for query in query_ids:
        query_skills = skills[query]
        relevance = np.zeros(len(df), dtype=bool)
        for candidate, candidate_skills in enumerate(skills):
            if candidate != query and query_skills and candidate_skills:
                jaccard = len(query_skills & candidate_skills) / len(
                    query_skills | candidate_skills
                )
                relevance[candidate] = jaccard >= relevance_threshold
        relevant_count = int(relevance.sum())
        if relevant_count == 0:
            continue
        scores = linear_kernel(matrix[query], matrix).ravel()
        scores[query] = -np.inf
        ranked = np.argpartition(scores, -max_k)[-max_k:]
        ranked = ranked[np.argsort(scores[ranked])[::-1]]
        for k in ks:
            hits = int(relevance[ranked[:k]].sum())
            rows.append({
                'query': query, 'k': k, 'precision': hits / k,
                'recall': hits / relevant_count
            })
        if not example:
            example = {
                'query_index': int(query),
                'query_title': df.at[query, 'course_title'],
                'relevant_count': relevant_count
            }
    detail = pd.DataFrame(rows)
    summary = detail.groupby('k', as_index=False).agg(
        precision_at_k=('precision', 'mean'),
        recall_at_k=('recall', 'mean'),
        evaluated_queries=('query', 'nunique')
    )
    protocol = {
        **example, 'sampled_queries': len(query_ids),
        'evaluated_queries': int(detail['query'].nunique()),
        'relevance_threshold': relevance_threshold
    }
    return summary, protocol

## 2. Load and inspect the catalog

The cleaned dataset contains 3,424 courses and 11 columns.

In [ ]:
courses = load_courses(DATA_PATH)
print(f'Shape: {courses.shape[0]:,} courses × {courses.shape[1]} columns')
display(courses[[
    'course_id', 'course_title', 'university',
    'difficulty_level', 'course_rating_filled'
]].head())

In [ ]:
summary = pd.DataFrame({
    'Value': [
        len(courses),
        courses['course_title'].nunique(),
        courses['university'].nunique(),
        courses['course_rating'].isna().sum(),
    ]
}, index=[
    'Courses', 'Unique titles', 'Universities / organizations',
    'Missing original ratings'
])
display(summary)

## 3. Feature preparation

The title is repeated three times and skills twice to increase their influence. This is a simple, interpretable form of field weighting.

In [ ]:
combined_text = build_text(courses)
print('Example course:', courses.loc[0, 'course_title'])
print('Combined text preview:')
print(combined_text.iloc[0][:500] + ' ...')

## 4. Fit TF-IDF

Key settings: English stop words, unigrams and bigrams, `min_df=2`, `max_df=0.95`, sublinear term frequency, and at most 30,000 features.

In [ ]:
vectorizer, course_matrix = fit_model(courses)
print('TF-IDF matrix shape:', course_matrix.shape)
print('Non-zero values:', f'{course_matrix.nnz:,}')
print('Sample features:', vectorizer.get_feature_names_out()[:20])

## 5. Generate Top-N recommendations

Change `QUERY_TITLE` or `TOP_N`, then run the cell. The function removes the query itself before ranking results.

In [ ]:
QUERY_TITLE = 'AI For Everyone'
TOP_N = 5

recommendations = recommend_by_title(
    courses, course_matrix, QUERY_TITLE, n=TOP_N
)
display(recommendations[[
    'rank', 'course_title', 'university',
    'difficulty_level', 'course_rating_filled', 'score'
]])

### Try another topic

In [ ]:
recommend_by_title(
    courses, course_matrix, 'Creative Writing', n=5
)[[
    'rank', 'course_title', 'university', 'score'
]]

## 6. Offline evaluation

Because the catalog has no learner behavior, another course is treated as relevant when its skill-set Jaccard similarity to the query is at least 0.20. A fixed seed samples 250 queries; 157 have at least one relevant neighbor.

In [ ]:
metrics, protocol = evaluate(
    courses, course_matrix,
    ks=(5, 10, 20),
    sample_size=250,
    relevance_threshold=0.20,
)
display(metrics.round(3))
protocol

In [ ]:
ax = metrics.set_index('k')[[
    'precision_at_k', 'recall_at_k'
]].plot(
    kind='bar', figsize=(8, 4.5),
    color=['#1B4965', '#5FA8D3']
)
ax.set(
    title='Offline Retrieval Performance',
    xlabel='K', ylabel='Mean score', ylim=(0, 1)
)
ax.legend(['Precision@K', 'Recall@K'], frameon=False)
plt.tight_layout()
plt.show()

## 7. Interpretation and limitations

- Precision decreases as the recommendation list becomes longer.
- Recall reaches approximately 85.5% at K = 20.
- The evaluation measures consistency with skill metadata, not learner satisfaction.
- Some lexical matches can cause semantic drift.
- Future work should use real clicks, enrollments, completions, ratings, and A/B testing.

## Short answers for questions

**Where is training?** `fit_model()` calls `fit_transform()` to learn TF-IDF vocabulary and IDF weights; there is no supervised training.  
**Why not collaborative filtering?** The dataset has no learner-course interaction matrix.  
**Why cosine similarity?** It works efficiently with sparse TF-IDF vectors and is less affected by text length.  
**Why does recall increase?** A longer list retrieves more of the relevant catalog neighbors.